# Scheduled ETL Pipeline for Retail Store Sales Data

This notebook implements an automated ETL (Extract, Transform, Load) pipeline that processes `retail_store_sales.csv` every 2 minutes using the `schedule` library.

In [6]:
# Import required libraries
import pandas as pd
import schedule
import time
from datetime import datetime
import os

## ETL Function Definition

The ETL function performs three main operations:
1. **Extract**: Read data from the CSV file
2. **Transform**: Clean and process the data
3. **Load**: Save the processed data to a new file

In [7]:
def etl_process():
    """
    ETL Pipeline: Extract, Transform, Load
    Runs every 2 minutes to process retail store sales data
    """
    try:
        print(f"\n{'='*60}")
        print(f"ETL Process Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*60}")
        
        # ==================== EXTRACT ====================
        print("\n[EXTRACT] Reading data from retail_store_sales.csv...")
        df = pd.read_csv('retail_store_sales.csv')
        print(f"✓ Successfully loaded {len(df)} records")
        print(f"  Columns: {list(df.columns)}")
        
        # ==================== TRANSFORM ====================
        print("\n[TRANSFORM] Cleaning and transforming data...")
        
        # 1. Handle missing values
        print("  - Handling missing values...")
        initial_rows = len(df)
        
        # Fill missing Item with 'Unknown'
        df['Item'].fillna('Unknown', inplace=True)
        
        # Fill missing Price Per Unit with median price by category
        df['Price Per Unit'].fillna(df.groupby('Category')['Price Per Unit'].transform('median'), inplace=True)
        
        # Fill missing Quantity with 1
        df['Quantity'].fillna(1.0, inplace=True)
        
        # Fill missing Total Spent: calculate from Price Per Unit * Quantity
        df['Total Spent'].fillna(df['Price Per Unit'] * df['Quantity'], inplace=True)
        
        # Fill missing Discount Applied with False
        df['Discount Applied'].fillna('False', inplace=True)
        
        # Drop rows with missing critical values (Category)
        df.dropna(subset=['Category'], inplace=True)
        print(f"    ✓ Removed {initial_rows - len(df)} rows with critical missing values")
        
        # 2. Data type conversions
        print("  - Converting data types...")
        df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
        df['Quantity'] = df['Quantity'].astype(float)
        df['Price Per Unit'] = df['Price Per Unit'].astype(float)
        df['Total Spent'] = df['Total Spent'].astype(float)
        
        # 3. Create additional features
        print("  - Creating additional features...")
        df['Year'] = df['Transaction Date'].dt.year
        df['Month'] = df['Transaction Date'].dt.month
        df['Day'] = df['Transaction Date'].dt.day
        df['DayOfWeek'] = df['Transaction Date'].dt.day_name()
        
        # 4. Remove duplicates
        initial_rows = len(df)
        df.drop_duplicates(subset=['Transaction ID'], keep='first', inplace=True)
        print(f"    ✓ Removed {initial_rows - len(df)} duplicate records")
        
        # 5. Add processing metadata
        df['Processed_Timestamp'] = datetime.now()
        df['Processing_Batch'] = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        print(f"  ✓ Transformation complete. Final record count: {len(df)}")
        
        # ==================== LOAD ====================
        print("\n[LOAD] Saving processed data...")
        
        # Create output directory if it doesn't exist
        output_dir = 'processed_data'
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"  - Created directory: {output_dir}/")
        
        # Save to CSV with timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = f'{output_dir}/processed_retail_sales_{timestamp}.csv'
        df.to_csv(output_file, index=False)
        print(f"  ✓ Saved processed data to: {output_file}")
        
        # Save latest version (overwrite)
        latest_file = f'{output_dir}/latest_processed_retail_sales.csv'
        df.to_csv(latest_file, index=False)
        print(f"  ✓ Updated latest version: {latest_file}")
        
        # Generate summary statistics
        print("\n[SUMMARY] Key Statistics:")
        print(f"  - Total Records Processed: {len(df)}")
        print(f"  - Total Revenue: ${df['Total Spent'].sum():,.2f}")
        print(f"  - Average Transaction Value: ${df['Total Spent'].mean():,.2f}")
        print(f"  - Date Range: {df['Transaction Date'].min()} to {df['Transaction Date'].max()}")
        print(f"  - Categories: {df['Category'].nunique()}")
        print(f"  - Unique Customers: {df['Customer ID'].nunique()}")
        
        print(f"\n{'='*60}")
        print(f"ETL Process Completed Successfully at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*60}\n")
        
    except Exception as e:
        print(f"\n[ERROR] ETL Process Failed: {str(e)}")
        print(f"{'='*60}\n")

## Schedule Configuration

Configure the scheduler to run the ETL process every 2 minutes.

In [8]:
# Schedule the ETL process to run every 2 minutes
schedule.every(2).minutes.do(etl_process)

print("📅 Scheduler Configured")
print("⏰ ETL process will run every 2 minutes")
print("🔄 Schedule details:")
print(f"   - Interval: 2 minutes")
print(f"   - Next run scheduled for: {schedule.next_run()}")
print("\n" + "="*60)

📅 Scheduler Configured
⏰ ETL process will run every 2 minutes
🔄 Schedule details:
   - Interval: 2 minutes
   - Next run scheduled for: 2026-02-10 11:26:22.313312



## Test Run

Run the ETL process once immediately to verify it works correctly.

In [9]:
# Run the ETL process once immediately as a test
print("🚀 Running initial ETL process...")
etl_process()

🚀 Running initial ETL process...

ETL Process Started at: 2026-02-10 11:26:09

[EXTRACT] Reading data from retail_store_sales.csv...
✓ Successfully loaded 12575 records
  Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']

[TRANSFORM] Cleaning and transforming data...
  - Handling missing values...
    ✓ Removed 0 rows with critical missing values
  - Converting data types...
  - Creating additional features...
    ✓ Removed 0 duplicate records
  ✓ Transformation complete. Final record count: 12575

[LOAD] Saving processed data...


C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item'].fillna('Unknown', inplace=True)
C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

  ✓ Saved processed data to: processed_data/processed_retail_sales_20260210_112609.csv
  ✓ Updated latest version: processed_data/latest_processed_retail_sales.csv

[SUMMARY] Key Statistics:
  - Total Records Processed: 12575
  - Total Revenue: $1,552,071.00
  - Average Transaction Value: $129.65
  - Date Range: 2022-01-01 00:00:00 to 2025-01-18 00:00:00
  - Categories: 8
  - Unique Customers: 25

ETL Process Completed Successfully at: 2026-02-10 11:26:10



## Start Scheduler

Run the scheduler continuously. The ETL process will execute automatically every 2 minutes.

**Note**: 
- This cell will run indefinitely. 
- To stop it, use the stop button in the notebook or press `Ctrl+C`
- You can set `max_runs` to limit the number of executions (useful for testing)

In [10]:
# Start the scheduler
print("🔄 Starting scheduler...")
print("⏰ ETL process will run every 2 minutes")
print("⛔ Press the Stop button or Ctrl+C to stop the scheduler\n")

# Option 1: Run for a limited number of times (good for testing)
# Uncomment the following lines and set max_runs to test
max_runs = 3  # Set to None for infinite runs
run_count = 0

try:
    while True:
        schedule.run_pending()
        time.sleep(1)
        
        # Check if we've reached max_runs (if set)
        if max_runs is not None and run_count >= max_runs:
            print(f"\n✅ Reached maximum runs ({max_runs}). Stopping scheduler.")
            break
            
        # Increment counter after each scheduled run
        if schedule.jobs and len(schedule.jobs) > 0:
            # Check if a job just ran by comparing next_run times
            pass
            
except KeyboardInterrupt:
    print("\n\n⛔ Scheduler stopped by user")
    print("Goodbye! 👋")

🔄 Starting scheduler...
⏰ ETL process will run every 2 minutes
⛔ Press the Stop button or Ctrl+C to stop the scheduler


ETL Process Started at: 2026-02-10 11:26:23

[EXTRACT] Reading data from retail_store_sales.csv...
✓ Successfully loaded 12575 records
  Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']

[TRANSFORM] Cleaning and transforming data...
  - Handling missing values...
    ✓ Removed 0 rows with critical missing values
  - Converting data types...
  - Creating additional features...
    ✓ Removed 0 duplicate records
  ✓ Transformation complete. Final record count: 12575

[LOAD] Saving processed data...


C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item'].fillna('Unknown', inplace=True)
C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

  ✓ Saved processed data to: processed_data/processed_retail_sales_20260210_112623.csv
  ✓ Updated latest version: processed_data/latest_processed_retail_sales.csv

[SUMMARY] Key Statistics:
  - Total Records Processed: 12575
  - Total Revenue: $1,552,071.00
  - Average Transaction Value: $129.65
  - Date Range: 2022-01-01 00:00:00 to 2025-01-18 00:00:00
  - Categories: 8
  - Unique Customers: 25

ETL Process Completed Successfully at: 2026-02-10 11:26:24


ETL Process Started at: 2026-02-10 11:28:09

[EXTRACT] Reading data from retail_store_sales.csv...
✓ Successfully loaded 12575 records
  Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']

[TRANSFORM] Cleaning and transforming data...
  - Handling missing values...
    ✓ Removed 0 rows with critical missing values
  - Converting data types...
  - Creating additional features...
    ✓ Removed 0 duplicate reco

C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item'].fillna('Unknown', inplace=True)
C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

  ✓ Saved processed data to: processed_data/processed_retail_sales_20260210_112809.csv
  ✓ Updated latest version: processed_data/latest_processed_retail_sales.csv

[SUMMARY] Key Statistics:
  - Total Records Processed: 12575
  - Total Revenue: $1,552,071.00
  - Average Transaction Value: $129.65
  - Date Range: 2022-01-01 00:00:00 to 2025-01-18 00:00:00
  - Categories: 8
  - Unique Customers: 25

ETL Process Completed Successfully at: 2026-02-10 11:28:09


ETL Process Started at: 2026-02-10 11:28:24

[EXTRACT] Reading data from retail_store_sales.csv...
✓ Successfully loaded 12575 records
  Columns: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']

[TRANSFORM] Cleaning and transforming data...
  - Handling missing values...
    ✓ Removed 0 rows with critical missing values
  - Converting data types...
  - Creating additional features...
    ✓ Removed 0 duplicate reco

C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Item'].fillna('Unknown', inplace=True)
C:\Users\mausa\AppData\Local\Temp\ipykernel_23368\3212461050.py:28: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

  ✓ Saved processed data to: processed_data/processed_retail_sales_20260210_112825.csv
  ✓ Updated latest version: processed_data/latest_processed_retail_sales.csv

[SUMMARY] Key Statistics:
  - Total Records Processed: 12575
  - Total Revenue: $1,552,071.00
  - Average Transaction Value: $129.65
  - Date Range: 2022-01-01 00:00:00 to 2025-01-18 00:00:00
  - Categories: 8
  - Unique Customers: 25

ETL Process Completed Successfully at: 2026-02-10 11:28:25



⛔ Scheduler stopped by user
Goodbye! 👋
